# M1 Pro — Unified Memory Coherence Test



## Objective (Apple M1 Pro)

Validate Metal unified memory sharing between CPU and GPU. Measure coherence
latency and document fallback paths when the working set exceeds the
`recommendedMaxWorkingSetSize`.


In [ ]:

# Auto-detect platform — bail out cleanly if not M1 Pro.
import sys
sys.path.insert(0, "..")  # so we can import platform_check
from platform_check import assert_apple_silicon
assert_apple_silicon()


In [ ]:

import nexusrt
info = nexusrt.firmware.init("m1pro")
print(info)
assert info.vendor == "apple", "This notebook requires Apple Silicon"


In [ ]:

# Unified memory: the same pointer is readable by both CPU and GPU.
# Write from CPU, read back via NexusRT prefetch (no-op on Metal).
import nexusrt.memory as mem
import ctypes

SIZE = 1 << 20  # 1 MB
a = mem.alloc(SIZE)
print(f"Allocated {SIZE} bytes at 0x{a.ptr:x}")

# CPU write — directly through the pointer.
arr = (ctypes.c_uint8 * SIZE).from_address(a.ptr)
for i in range(SIZE): arr[i] = i & 0xFF

# NexusRT prefetch — on Metal this is a no-op (data is already visible to GPU).
mem.prefetch(a.ptr, SIZE)

# CPU read-back — verify coherence.
ok = all(arr[i] == (i & 0xFF) for i in range(0, SIZE, 4096))
print(f"Coherence: {'OK' if ok else 'FAIL'}")
mem.free(a)


In [ ]:

# Coherence latency: time a CPU→GPU→CPU round trip.
import time
N = 1000
a = mem.alloc(4096)
arr = (ctypes.c_uint8 * 4096).from_address(a.ptr)
t0 = time.perf_counter()
for _ in range(N):
    arr[0] = (arr[0] + 1) & 0xFF   # CPU write
    mem.prefetch(a.ptr, 4096)      # GPU visibility
    _ = arr[0]                      # CPU read
t1 = time.perf_counter()
print(f"Avg coherence RTT: {(t1-t0)*1e6/N:.2f} us")
mem.free(a)


In [ ]:

# Fallback path: working set exceeds recommendedMaxWorkingSetSize.
# On M1 Pro this is ~14 GB. We deliberately allocate beyond it and confirm
# NexusRT falls back gracefully.
try:
    huge = mem.alloc(15 << 30)   # 15 GB
    print("15 GB allocation succeeded — Metal allowed it.")
    mem.free(huge)
except Exception as e:
    print(f"15 GB allocation rejected (expected on smaller M1 Pro SKUs): {e}")


In [ ]:

nexusrt.firmware.shutdown()
print("✓ M1 Pro unified memory coherence test complete.")
